In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Code

In [4]:
from __future__ import annotations

import os
import pickle
import re
import unicodedata
from collections import Counter
from contextlib import contextmanager
from dataclasses import dataclass, field
from enum import Enum, auto
from pathlib import Path
from typing import Callable, Dict, Iterator, List, Optional, Tuple

import matplotlib as mpl
import matplotlib.patches
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.font_manager as fm
import seaborn as sns

import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoModelForMultimodalLM,
    AutoTokenizer,
    PreTrainedTokenizerBase,
)

# import koreanize_matplotlib

import unicodedata
from collections import Counter
from functools import lru_cache

from datasets import load_dataset, load_from_disk

import gc
import torch
import random

import re
import unicodedata
from collections import Counter
from functools import lru_cache
from typing import Dict, List, Optional
 
from transformers import PreTrainedTokenizerBase
from datasets import DatasetDict

In [5]:
from font_fallback import setup_fonts, init_font_fallback, draw_text_fb, annotate_heatmap_fb, sanitize_tok_str, draw_rotated_token

FONT_DIR = "../fonts/"

FONT_PRIORITY = [
    "Noto Sans",
    "Noto Sans KR",
    "Noto Sans JP",
    "Noto Sans SC",
    "Noto Sans Devanagari",
    "Noto Sans Gujarati",
    "Noto Sans Tamil",
    "Noto Sans Telugu",
    "Noto Sans Thai",
    "Noto Sans Arabic",
    "Noto Sans Symbols",
    "NanumGothic",
    "Noto Emoji",
    "DejaVu Sans",   # guaranteed fallback — keep last
]


setup_fonts(FONT_DIR)
init_font_fallback(FONT_PRIORITY)   
mpl.rcParams["font.family"]      = "sans-serif"
mpl.rcParams["font.sans-serif"]  = FONT_PRIORITY   # still useful for plain ax.set_title() etc.
mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams["figure.dpi"]  = 150
mpl.rcParams["savefig.dpi"] = 150


font_fallback: registered 342 font file(s) from '../fonts/'
font_fallback: priority list set (14 fonts)


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. Constants
# ─────────────────────────────────────────────────────────────────────────────

# Sub-layer slot indices inside layer_states[:, slot, :]
SLOT_POST_ATTN  = 0
SLOT_POST_MLP   = 1
SLOT_FULL_BLOCK = 2
SLOT_NAMES: Dict[int, str] = {
    SLOT_POST_ATTN:  "post-attn",
    SLOT_POST_MLP:   "post-MLP",
    SLOT_FULL_BLOCK: "full-block",
}

# plotting setup
PLOT_STYLE = {
    "title":      30, # 14,   # was ~9-11
    "axis":       20, # 13,   # was 8-9
    "tick":       20, # 11,   # was 5.5-8
    "legend":     20, # 11,   # was 7-8
    "annot":      10,   # heatmap cell text
    "strip":       9,   # colour-strip / top-1 token strip labels
    "colorbar":   11,
}

TICK_WEIGHT   = "bold"
LABEL_WEIGHT  = "bold"
TITLE_WEIGHT  = "bold"

# Language label constants
LANG_LATIN=      "Latin"
LANG_KOREAN=     "Hangul"
LANG_JAPANESE=   "Kana"
LANG_CHINESE=    "CJK"
LANG_ARABIC=     "Arabic"
LANG_CYRILLIC=   "Cyrillic"
LANG_GREEK=    "Greek"
LANG_HEBREW=     "Hebrew"
LANG_THAI=      "Thai"
LANG_DEVANAGARI= "Devanagari"
LANG_BENGALI=    "Bengali"
LANG_TELUGU=    "Telugu"
LANG_SPECIAL=    "Special"
LANG_OTHER=     "Other"
LANG_PARTIAL=    "Partial"  # byte fragment — script unknown


ALL_LANG_LABELS: List[str] = [
    LANG_LATIN, LANG_KOREAN, LANG_CHINESE, LANG_JAPANESE,
    LANG_ARABIC, LANG_CYRILLIC, LANG_GREEK, LANG_HEBREW,
    LANG_THAI, LANG_DEVANAGARI, LANG_BENGALI, 
    LANG_TELUGU, LANG_OTHER, LANG_SPECIAL, LANG_PARTIAL,
]

# Unicode-name → script mapping
SCRIPT_KEYWORDS: Dict[str, str] = {
    "HANGUL":      LANG_KOREAN,

    "HIRAGANA":    LANG_JAPANESE,
    "KATAKANA":    LANG_JAPANESE,

    "CJK":         LANG_CHINESE,
    "IDEOGRAPH":   LANG_CHINESE,
    "KANGXI":      LANG_CHINESE,
    "BOPOMOFO":    LANG_CHINESE,

    "LATIN":       LANG_LATIN,

    "ARABIC":      LANG_ARABIC,
    "CYRILLIC":    LANG_CYRILLIC,
    "GREEK":       LANG_GREEK,
    "HEBREW":      LANG_HEBREW,
    "THAI":        LANG_THAI,

    "DEVANAGARI":  LANG_DEVANAGARI,
    "BENGALI":     LANG_BENGALI,
    "TELUGU":      LANG_TELUGU,

    # recommended additions
    "HALFWIDTH KATAKANA": LANG_JAPANESE,
    "FULLWIDTH LATIN":    LANG_LATIN,
}


LANG_COLORS: Dict[str, str] = {
    # ── Major scripts ─────────────────────────────────────────────
    LANG_LATIN:       "#377EB8",   # vivid red
    LANG_CHINESE:     "#E41A1C",   # strong blue
    LANG_KOREAN:      "#FF7F00",   # vivid orange
    LANG_JAPANESE:    "#984EA3",   # purple

    # ── Middle-frequency scripts ─────────────────────────────────
    LANG_ARABIC:      "#4DAF4A",   # green
    LANG_CYRILLIC:    "#A65628",   # brown
    LANG_GREEK:       "#F781BF",   # pink
    LANG_HEBREW:      "#FFD92F",   # yellow
    LANG_THAI:        "#17BECF",   # cyan

    # ── Indic scripts ─────────────────────────────────────────────
    LANG_DEVANAGARI:  "#66A61E",   # olive green
    LANG_BENGALI:     "#E6AB02",   # mustard
    LANG_TELUGU:      "#1B9E77",   # sea green

    # ── Special buckets ───────────────────────────────────────────
    LANG_OTHER:       "#7F7F7F",   # neutral gray
    LANG_SPECIAL:     "#D9D9D9",   # light gray
    LANG_PARTIAL:     "#252525",   # near black
}

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. Unicode script classifier
# ─────────────────────────────────────────────────────────────────────────────
# SentencePiece byte-escape pattern: <0xHH>
_SP_BYTE_RE = re.compile(r"<0x([0-9A-Fa-f]{2})>")

# Special-token surface pattern (e.g. <pad>, <|im_start|>, [CLS])
_SPECIAL_TOKEN_RE = re.compile(
    r"^(?:"
    r"<\|?[a-zA-Z0-9_\-]+\|?>|"
    r"\[(?:PAD|UNK|CLS|SEP|MASK|BOS|EOS)\]|"
    r"</?s>|</s>|<s>"
    r")$"
)


def _build_tiktoken_byte_map() -> Dict[int, int]:
    """Return {unicode_ord → byte_value} for tiktoken byte-mapped tokens."""
    bs: List[int] = (
        list(range(ord("!"),  ord("~") + 1))
        + list(range(ord("¡"), ord("¬") + 1))
        + list(range(ord("®"), ord("ÿ") + 1))
    )
    cs = bs[:]
    n  = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return {c: b for b, c in zip(bs, cs)}


_TIKTOKEN_UNICODE_TO_BYTE: Dict[int, int] = _build_tiktoken_byte_map()


def _surface_to_bytes(surface: str) -> Optional[bytes]:
    """
    Convert a tokenizer surface string to a raw byte sequence.

    Handles three surface formats:
      1. SentencePiece byte escapes:  <0xE3><0x84><0xB9>
      2. tiktoken / Llama-3 byte-mapped characters
      3. Plain Unicode (including subword markers ▁ Ġ Ċ)

    Returns None for empty / special-token-only surfaces.
    """
    # Case 1: SentencePiece byte escapes
    if "<0x" in surface:
        raw = bytearray()
        pos = 0
        for m in _SP_BYTE_RE.finditer(surface):
            prefix = surface[pos:m.start()]
            if prefix:
                raw.extend(prefix.encode("utf-8"))
            raw.append(int(m.group(1), 16))
            pos = m.end()
        tail = surface[pos:]
        if tail:
            raw.extend(tail.encode("utf-8"))
        return bytes(raw)

    # Case 2: tiktoken byte-mapped characters
    clean = surface.lstrip("▁Ġ")
    if clean and all(ord(ch) in _TIKTOKEN_UNICODE_TO_BYTE for ch in clean):
        return bytes(_TIKTOKEN_UNICODE_TO_BYTE[ord(ch)] for ch in clean)

    # Case 3: Plain Unicode — strip subword-start markers
    plain = surface.lstrip("▁ĠĊ")
    if not plain:
        return b""
    return plain.encode("utf-8")



@lru_cache(maxsize=65536)
def _classify_char(ch: str) -> str:
    # Fast cached character classification
    # https://www.unicode.org/reports/tr44/tr44-34.html#General_Category_Values
    cat = unicodedata.category(ch)

    # whitespace/control/format
    if cat[0] in {"Z", "C"}:
        return LANG_SPECIAL

    name = unicodedata.name(ch, "")

    # emoji + pictographs
    if any(kw in name for kw in ("EMOJI", "FACE", "HAND", "EMOTICON")):
        return LANG_SPECIAL

    # ── script keyword lookup ──────────
    for keyword, label in SCRIPT_KEYWORDS.items():
        # if keyword in name and keyword == "LATIN":
        #     from langdetect import detect, detect_langs
        #     DetectorFactory.seed = 0
        #     label = detect_langs(ch)
        if keyword in name:
            return label

    # ── combining marks with no recognized script → neutral ──────────────────
    # Only reaches here if the Unicode name didn't match any SCRIPT_KEYWORDS
    # entry (e.g. generic diacritics such as U+0300 COMBINING GRAVE ACCENT).
    if cat.startswith("M"):
        return LANG_SPECIAL

    # punctuation / symbols / numbers
    if cat[0] in {"N", "P", "S"}:
        return LANG_SPECIAL

    return LANG_OTHER


def _classify_unicode_text(
    text: str,
    *,
    ignore_special: bool = True,
    min_vote_ratio: float = 0.20,
) -> str:
    """
    Coarse Unicode script classifier.

    Parameters
    ----------
    text
        Input Unicode text.

    ignore_special
        Ignore punctuation/symbol/control votes when
        script characters exist.

    min_vote_ratio
        Minimum winning vote ratio required to avoid
        returning LANG_OTHER.

    Returns
    -------
    str
        Dominant script/language label.
    """

    if not text or not text.strip():
        return LANG_SPECIAL

    votes = Counter()

    for ch in text:
        label = _classify_char(ch)
        votes[label] += 1

    if not votes:
        return LANG_SPECIAL

    # remove specials if real scripts exist
    if ignore_special:

        script_votes = {
            k: v
            for k, v in votes.items()
            if k not in {LANG_SPECIAL, LANG_OTHER}
        }

        if script_votes:
            votes = Counter(script_votes)

    total = sum(votes.values())

    if total == 0:
        return LANG_SPECIAL

    winner, count = votes.most_common(1)[0]

    # reject weak dominance
    if count / total < min_vote_ratio:
        return LANG_OTHER

    return winner


def identify_token_language(
    token_id:  int,
    tokenizer: PreTrainedTokenizerBase,
) -> str:
    """
    Identify the script/language family of a single tokenizer token.

    Returns one of ALL_LANG_LABELS.

    Algorithm
    ---------
    1. Known special token ids  → "special"
    2. convert_ids_to_tokens()  → surface string
    3. Angle-bracket special pattern → "special"
    4. _surface_to_bytes()      → raw bytes
    5. bytes.decode("utf-8")    → success: _classify_unicode_text()
                                  UnicodeDecodeError: "partial"
    """
    if token_id in tokenizer.all_special_ids:
        return LANG_SPECIAL

    surface: str = tokenizer.convert_ids_to_tokens(token_id)
    if surface is None:
        return LANG_SPECIAL

    if _SPECIAL_TOKEN_RE.match(surface):
        return LANG_SPECIAL

    raw_bytes = _surface_to_bytes(surface)
    if raw_bytes is None or raw_bytes == b"":
        return LANG_SPECIAL

    try:
        text = raw_bytes.decode("utf-8", errors="strict")
    except UnicodeDecodeError:
        return LANG_PARTIAL

    if not text.strip():
        return LANG_SPECIAL

    return _classify_unicode_text(text)


def build_token_language_map(
    tokenizer:  PreTrainedTokenizerBase,
    batch_size: int = 4096,   # kept for API compatibility; iteration is per-token
) -> Dict[int, str]:
    """
    Build a {token_id → language_label} mapping for the entire vocabulary.

    This is a one-time cost called inside build_analyzer().
    """
    vocab_size = tokenizer.vocab_size
    return {
        tid: identify_token_language(tid, tokenizer)
        for tid in range(vocab_size)
    }

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.  Data container
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class SampleAnly:
    """
    One analysed sample with sub-layer hidden states.

    Hidden state layout
    -------------------
    embedding_state : Tensor (H,)
        Output of the token embedding lookup (before any transformer block).

    layer_states : Tensor (L, 3, H)
        Three checkpoints per transformer layer:
          [:, 0, :]  post-attention  — residual stream after self-attention
          [:, 1, :]  post-MLP        — residual stream after MLP
          [:, 2, :]  full-block      — alias for post-MLP (stored separately
                                       for clarity; always equals [:, 1, :]) # ------------> 시각화 결과 다르게 뽑힘. 확인필요

    Derived views
    -------------
    all_states : Tensor (1 + L*3, H)
        Flat concatenation: [embedding, l0_attn, l0_mlp, l0_block, l1_attn, …]

    Result fields (filled by LogitLensAnalyzer.process_hidden_states)
    -----------------------------------------------------------------
    max_prob_token_ids : Tensor (1 + L*3,)
        Argmax token id at each checkpoint.

    top1_lang_ids      : Tensor (1 + L*3,)
        Index into ALL_LANG_LABELS for the top-1 token at each checkpoint.
    """

    sample_id:     str
    question:      str
    answer:        str   # Free-form answer (open-ended datasets)
    prompt: str

    # Core hidden states — set by capture pipeline
    embedding_state: torch.Tensor          # (H,)
    layer_states:    torch.Tensor          # (L, 3, H)
    logits:    torch.Tensor         
    
    # Filled by process_hidden_states()
    max_prob_token_ids: torch.Tensor = field(default_factory=lambda: torch.tensor([]))
    top1_lang_ids:      torch.Tensor = field(default_factory=lambda: torch.tensor([]))

    # Dataset-level metadata
    language:     Optional[str] = None   # ISO 639-3, e.g. "kor"
    # script:       Optional[str] = None   # ISO 15924, e.g. "Hang"
    # dataset_name: Optional[str] = None
    # task_type:    Optional[str] = None
    
    # ── Convenience properties ────────────────────────────────────────────────
    @property
    def n_layers(self) -> int:
        return self.layer_states.shape[0]

    @property
    def n_checkpoints(self) -> int:
        """Total unembedding checkpoints: 1 embedding + L × 3 sub-layer slots."""
        return 1 + self.n_layers * 3

    @property
    def all_states(self) -> torch.Tensor:
        """(1 + L*3, H) flat view for unembedding."""
        L, _, H = self.layer_states.shape
        flat_layers = self.layer_states.reshape(L * 3, H)
        return torch.cat([self.embedding_state.unsqueeze(0), flat_layers], dim=0)

    @staticmethod
    def checkpoint_labels(n_layers: int) -> List[str]:
        """
        Human-readable label for every checkpoint position.

        Returns list of length 1 + n_layers*3:
            ["emb", "L0-attn", "L0-mlp", "L0-blk", "L1-attn", …]
        """
        labels = ["emb"]
        for li in range(n_layers):
            labels += [f"L{li}-attn", f"L{li}-mlp", f"L{li}-blk"]
        return labels

    @property
    def cp_labels(self) -> List[str]:
        return self.checkpoint_labels(self.n_layers)


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.  Batched forward-hook infrastructure
# ─────────────────────────────────────────────────────────────────────────────

class BatchedHiddenStateCapture:
    """
    Attaches forward hooks that store the full (B, T, H) hidden-state tensor
    for each sub-layer checkpoint, then reduces to (B, H) per checkpoint by
    selecting the last token of each sequence via the attention mask. ------> literal last token 뽑는 쪽으로 수정?

    Hook storage layout
    -------------------
    _raw["emb"]        : Tensor (B, T, H)
    _raw["L{i}_attn"]  : Tensor (B, T, H)
    _raw["L{i}_mlp"]   : Tensor (B, T, H)
    _raw["L{i}_block"] : Tensor (B, T, H)

    Usage
    -----
        capture = BatchedHiddenStateCapture(model)
        with capture.record():
            _ = model(input_ids=ids, attention_mask=mask)
        emb, ls = capture.get_last_token_states(mask.cpu())
        capture.clear()

    Supported architectures
    -----------------------
    Llama 3.x     : model.model.embed_tokens  /  model.model.layers[i]
    Gemma3/Qwen3.5: model.model.language_model.embed_tokens  / .layers[i]
    """

    def __init__(self, model: nn.Module, padding_type: str):
        self.model  = model
        self.padding_type = padding_type
        self._hooks: List[torch.utils.hooks.RemovableHook] = []
        self._raw:  Dict[str, torch.Tensor] = {}
        self._embed_module, self._layer_modules = self._locate_modules(model)

    # ── Module discovery ──────────────────────────────────────────────────────

    @staticmethod
    def _locate_modules(
        model: nn.Module,
    ) -> Tuple[nn.Module, List[Tuple[nn.Module, nn.Module, nn.Module]]]:
        multimodal_prefixes = ("gemma3", "gemma4", "qwen3_5")
        if any(model.config.model_type.startswith(p) for p in multimodal_prefixes):
            lm = model.model.language_model
        else:
            lm = model.model

        embed  = lm.embed_tokens
        triples = []
        for layer in lm.layers:
            attn = getattr(layer, "self_attn", None) or getattr(layer, "linear_attn") # qwen3.5: switch between linear_attn, self_attn
            triples.append((attn, layer.mlp, layer))
        return embed, triples

    # ── Hook builder ──────────────────────────────────────────────────────────

    def _make_hook(self, key: str):
        def hook(module, input, output):
            if isinstance(output, torch.Tensor):
                hs = output
            elif isinstance(output, (tuple, list)):
                hs = output[0]
            else:
                hs = output.last_hidden_state

            if hs.ndim != 3:
                raise ValueError(
                    f"Hook '{key}': expected (B, T, H), got {tuple(hs.shape)}"
                )
            # Move to CPU immediately to avoid holding VRAM across layers
            self._raw[key] = hs.detach().cpu()   # (B, T, H)
        return hook

    # ── Context manager ───────────────────────────────────────────────────────

    @contextmanager
    def record(self) -> Iterator[None]:
        """Attach hooks for one forward pass, then remove them."""
        h = self._embed_module.register_forward_hook(self._make_hook("emb"))
        self._hooks.append(h)

        for li, (attn, mlp, block) in enumerate(self._layer_modules):
            for sub, slot in [(attn, "attn"), (mlp, "mlp"), (block, "block")]:
                h = sub.register_forward_hook(self._make_hook(f"L{li}_{slot}"))
                self._hooks.append(h)
        try:
            yield
        finally:
            for h in self._hooks:
                h.remove()
            self._hooks.clear()

    # ── Last-token extraction ─────────────────────────────────────────────────

    def get_last_token_states(
        self,
        attention_mask: torch.Tensor,            # (B, T) — 1=real, 0=pad
        dtype: torch.dtype = torch.float32,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Reduce (B, T, H) tensors to (B, H) by selecting the last real token
        of each sequence (correct for both left- and right-padded inputs).

        Returns
        -------
        embedding_states : Tensor (B, H)
        layer_states     : Tensor (B, L, 3, H)
        """
        # --------------> batch processing + left padding
        mask = attention_mask.cpu()
        B, T = mask.shape

        # ------------- Find last token
        if self.padding_type == "last-non-pad":
            # Find last non-padding token index
            last_pos = T - 1 - mask.flip(dims=[1]).argmax(dim=1) # read mask backwards -> get the last non-padding token idx
        
        elif self.padding_type == "last-literal":
            # Literal last token position (works for left padding)
            last_pos = torch.full(
                (B,),
                T - 1,
                device=mask.device,
                dtype=torch.long,
            )
        
        else:
            raise ValueError(f"Unknown padding_type: {self.padding_type}")
        # ------------- 

        def _pick(key: str) -> torch.Tensor: # pick last token
            hs  = self._raw[key]              # (B, T, H)
            H   = hs.shape[2]  # hidden dim
            idx = last_pos.view(B, 1, 1).expand(B, 1, H)   # (B, 1, H)
            return hs.gather(dim=1, index=idx).squeeze(1).to(dtype)  # (B, H)

        embedding_states = _pick("emb")       # (B, H)
        L = len(self._layer_modules)
        H = embedding_states.shape[1]
        layer_states = torch.zeros(B, L, 3, H, dtype=dtype)

        for li in range(L):
            layer_states[:, li, SLOT_POST_ATTN,  :] = _pick(f"L{li}_attn") # (B, H)
            layer_states[:, li, SLOT_POST_MLP,   :] = _pick(f"L{li}_mlp")
            layer_states[:, li, SLOT_FULL_BLOCK, :] = _pick(f"L{li}_block")

        return embedding_states, layer_states  # (B, H), (B, L, 3, H)

    def clear(self) -> None:
        """Discard all captured tensors between batches."""
        self._raw.clear()
        # ADD: break circular refs from closures that captured `self`
        self._hooks.clear()

    def __del__(self):
        """Ensure hooks are removed if capture object is garbage collected."""
        for h in self._hooks:
            try:
                h.remove()
            except Exception:
                pass
        self._hooks.clear()
        self._raw.clear()

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# 5.  Batched capture pipeline
# ─────────────────────────────────────────────────────────────────────────────

def capture_hidden_states_batched(
    model:              nn.Module,
    tokenizer:          PreTrainedTokenizerBase,
    dataset:            List[Dict],
    padding_type: str,
    device:             str = "cuda",
    capture_batch_size: int = 8,
    max_length:         int = 512,
    out_path:           Optional[str | Path] = None,
    dtype:              torch.dtype = torch.float32,
) -> List[Dict]:
    """
    Run the model over a list of prompt dicts and capture sub-layer hidden
    states via batched forward hooks.

    Parameters
    ----------
    model : nn.Module
        Loaded HuggingFace causal / multimodal LM (already on `device`).
    tokenizer : PreTrainedTokenizerBase
        Must support padding.  If no pad_token is set, eos_token is used.
    dataset : list[dict]
        Each dict must contain at minimum:
            "sample_id", "question", "answer"
        Optional keys ("language", "script", …) are forwarded
        verbatim to output dicts.
    device : str
    capture_batch_size : int
        Prompts per GPU forward pass.  Reduce if OOM.
    max_length : int
        Tokenizer truncation length.  Sequences are truncated from the LEFT
        so that the end of the prompt (question and answer choices) is always
        preserved in the last token's receptive field.
    out_path : str | Path, optional
        If given, output list[dict] is pickled here.
    dtype : torch.dtype
        Storage dtype for hidden states.  float32 is recommended for
        numerical stability; bfloat16 halves memory.

    Returns
    -------
    list[dict]  — same order as `dataset`, each dict extended with:
        "embedding_state" : np.ndarray (H,)
        "layer_states"    : np.ndarray (L, 3, H)
    """
    if tokenizer.pad_token is None:
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    # Left-padding ensures the last real token is always at position T-1,
    # which makes last-token extraction trivial and avoids right-pad artefacts.
    original_padding_side   = tokenizer.padding_side
    tokenizer.padding_side  = "left"

    capture = BatchedHiddenStateCapture(model, padding_type)
    model.eval()

    results: List[Dict] = []
    prompts = [item["prompt"] for item in dataset] # input string formatted w/ tokenizer's chat template.
    n       = len(prompts)

    try:
        for start in tqdm(range(0, n, capture_batch_size), desc="capture_hidden_states"):
            end           = min(start + capture_batch_size, n)
            batch_prompts = prompts[start:end]
            batch_items   = dataset[start:end]

            encoding = tokenizer(
                batch_prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_length,
            )
            
            input_ids      = encoding["input_ids"].to(device)       # (B, T)
            attention_mask = encoding["attention_mask"].to(device)  # (B, T)

            with torch.no_grad(), capture.record():
                _ = model(input_ids=input_ids, attention_mask=attention_mask)

            emb_batch, ls_batch = capture.get_last_token_states(
                attention_mask.cpu(), dtype=dtype
            )  # (B, H), (B, L, 3, H)
            capture.clear()

            for b, item in enumerate(batch_items):
                out = {k: v for k, v in item.items()
                       if k not in ("embedding_state", "layer_states")}
                out["embedding_state"] = emb_batch[b].numpy()   # (H,)
                out["layer_states"]    = ls_batch[b].numpy()    # (L, 3, H)
                results.append(out)

    finally:
        tokenizer.padding_side = original_padding_side
    
    if out_path is not None:
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        with open(out_path, "wb") as f:
            pickle.dump(results, f)
        print(f"Saved {len(results)} samples → {out_path}")

    return results

In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# 7.  Analyser
# ─────────────────────────────────────────────────────────────────────────────

class LogitLensAnalyzer:
    """
    Unembeds hidden states at every checkpoint and computes multilingual
    token-level and population-level statistics.

    Parameters
    ----------
    norm_layer       : model's final normalisation layer
    lm_head          : unembedding linear layer
    token_lang_map   : {token_id → language_label}; build with
                       build_token_language_map(tokenizer)
    device           : inference device (default: lm_head's device)
    batch_size       : samples per GPU batch in process_hidden_states()
    """

    def __init__(
        self,
        norm_layer:       nn.Module,
        lm_head:          nn.Module,
        token_lang_map:   Dict[int, str],
        device:           Optional[str | torch.device] = None,
        batch_size:       int = 32,
    ):
        self.norm_layer  = norm_layer
        self.lm_head     = lm_head
        self.device      = device or next(lm_head.parameters()).device
        self.batch_size  = batch_size
        self.token_lang_map = token_lang_map

        # Precompute lang_id lookup: lang_id_lookup[token_id] = index in ALL_LANG_LABELS
        label2idx   = {lbl: i for i, lbl in enumerate(ALL_LANG_LABELS)}
        vocab_size  = max(token_lang_map.keys()) + 1 # key: token idx
        lang_id_arr = torch.full(
            (vocab_size,), label2idx[LANG_OTHER], dtype=torch.long 
        ) # torch.full(size, fill_value) # LANG_OTHER
        
        for tid, lbl in token_lang_map.items(): # token id : label (lang)
            if tid < vocab_size:
                lang_id_arr[tid] = label2idx.get(lbl, label2idx[LANG_OTHER])
        self.lang_id_lookup = lang_id_arr   # (vocab_size,) — stays on CPU

        self.norm_layer.eval()
        self.lm_head.eval()
        for p in (*self.norm_layer.parameters(), *self.lm_head.parameters()):
            p.requires_grad_(False)

    # ── I/O ──────────────────────────────────────────────────────────────────

    def load_samples(
        self,
        path_pkl: str | Path,
        dtype:    torch.dtype,
    ) -> List[SampleAnly]:
        """
        Load pickled hidden states into SampleAnly objects.

        Accepted dict schemas
        ---------------------
        Sub-layer format (preferred):
            "embedding_state" : ndarray (H,)
            "layer_states"    : ndarray (L, 3, H)

        Legacy format (single vector per layer):
            "n_layer_hidden_states" : ndarray (L+1, H)
            (embedding is hs[0]; layer states are broadcast to 3 slots)

        Optional keys forwarded verbatim:
            "language", "script", "dataset_name", "task_type"
        """
        with open(path_pkl, "rb") as f:
            raw = pickle.load(f)

        samples: List[SampleAnly] = []
        for item in raw:
            if "layer_states" in item:
                emb = torch.as_tensor(item["embedding_state"]).to(dtype)
                ls  = torch.as_tensor(item["layer_states"]).to(dtype)
            else:
                raise ValueError(
                    "Dict must contain 'layer_states'."
                )

            samples.append(SampleAnly(
                sample_id=item["sample_id"],
                question=item["question"],
                answer=item.get("answer", ""),
                prompt=item["prompt"],
                logits=item.get("logits", torch.tensor([])),
                language=item.get("language"),
                
                embedding_state=emb,
                layer_states=ls,
            ))
        return samples

    # ── Core unembedding ──────────────────────────────────────────────────────

    @torch.no_grad()
    def unembed(self, hidden_states: torch.Tensor) -> torch.Tensor:
        """norm → lm_head.  Input: (..., H)  Output: (..., vocab_size)."""
        return self.lm_head(self.norm_layer(hidden_states.to(self.device)))

    # ── Main processing loop ──────────────────────────────────────────────────

    @torch.no_grad()
    def process_hidden_states(self, samples: List[SampleAnly]) -> None:
        n = len(samples)
        all_hs = torch.stack([s.all_states for s in samples])   # (N, 1+3*L, H)
    
        all_logits:   List[torch.Tensor] = []
        all_max_ids:  List[torch.Tensor] = []
        all_lang_ids: List[torch.Tensor] = []
    
        for start in tqdm(range(0, n, self.batch_size), desc="process_hidden_states"):
            end   = min(start + self.batch_size, n)
            batch = all_hs[start:end].to(self.device)     # (B, C, H)
            B, C, H = batch.shape
    
            logits = self.unembed(batch.view(B * C, H)).view(B, C, -1)  # (B, C, V)
    
            # --- KEY CHANGE: argmax directly on logits, no softmax needed ---
            # argmax(logits) == argmax(softmax(logits)), so probs are unnecessary.
            max_ids = logits.argmax(dim=-1).cpu()                        # (B, C)
    
            clamp_max = self.lang_id_lookup.shape[0] - 1
            lang_ids  = self.lang_id_lookup[max_ids.clamp(max=clamp_max)]  # (B, C)
    
            all_logits.append(logits.cpu())   # move off GPU immediately
            all_max_ids.append(max_ids)
            all_lang_ids.append(lang_ids)
    
            # Explicitly free GPU tensors before next batch
            del logits, batch
            torch.cuda.empty_cache()
    
        logits_all   = torch.cat(all_logits,   dim=0)   # (N, C, V)
        max_ids_all  = torch.cat(all_max_ids,  dim=0)   # (N, C)
        lang_ids_all = torch.cat(all_lang_ids, dim=0)   # (N, C)
    
        for i, sample in enumerate(samples):
            sample.logits            = logits_all[i]
            sample.max_prob_token_ids = max_ids_all[i]
            sample.top1_lang_ids      = lang_ids_all[i]

        
    # ── Public metric methods — language ──────────────────────────────────────

    def layerwise_lang_proportions(
        self,
        samples: List[SampleAnly],
    ) -> np.ndarray:
        """
        Proportion of each language label among top-1 tokens at every
        checkpoint, averaged over all samples.

        Returns
        -------
        proportions : np.ndarray (C, n_lang_labels)
            Each row sums to 1.0.  Column order matches ALL_LANG_LABELS.
        """
        n_lang   = len(ALL_LANG_LABELS)
        C        = samples[0].n_checkpoints
        lang_ids = torch.stack([s.top1_lang_ids for s in samples])  # (N, C)

        proportions = np.zeros((C, n_lang), dtype=np.float32)  # (C, n_lang)
        for c in range(C):
            col = lang_ids[:, c] # c라는 레이어에서, 전체 샘플에 대한 lang_ids
            for li in range(n_lang):
                proportions[c, li] = (col == li).float().mean().item() # match_num (=해당 언어로 매칭된 개수) / total_Num(= 전체 sample 개수)
        return proportions  # (C, n_lang)

    def lang_switching_rate(self, samples: List[SampleAnly]) -> np.ndarray:
        """
        (C-1,) fraction of samples where the top-1 token's language label
        changes between consecutive checkpoints.  Reveals when the model
        transitions between scripts during processing.
        """
        lang_ids = torch.stack([s.top1_lang_ids for s in samples])  # (N, C)
        switches = (lang_ids[:, 1:] != lang_ids[:, :-1]).float()    # (N, C-1)
        return switches.mean(dim=0).numpy()                          # (C-1,) # dim=0 -> vertical average.

    def dominant_lang_per_checkpoint(
        self,
        samples: List[SampleAnly],
    ) -> List[str]:
        """
        The most common top-1 token language at each checkpoint.

        Returns list of length C (one label per checkpoint).
        """
        proportions = self.layerwise_lang_proportions(samples)   # (C, n_lang)
        return [ALL_LANG_LABELS[int(np.argmax(proportions[c]))] # List (len: C)
                for c in range(proportions.shape[0])]

    
    # ── Internal helpers ───────────────────────────────────────────────────────────
    
    def _apply_axis_style(self, ax: plt.Axes) -> None:
        """Apply unified font sizes + bold weights to a finished axes."""
        ax.tick_params(axis="both", labelsize=PLOT_STYLE["tick"])
        for lbl in ax.get_xticklabels() + ax.get_yticklabels():
            lbl.set_fontweight(TICK_WEIGHT)
        ax.title.set_fontsize(PLOT_STYLE["title"])
        ax.title.set_fontweight(TITLE_WEIGHT)
        ax.xaxis.label.set_size(PLOT_STYLE["axis"])
        ax.xaxis.label.set_fontweight(LABEL_WEIGHT)
        ax.yaxis.label.set_size(PLOT_STYLE["axis"])
        ax.yaxis.label.set_fontweight(LABEL_WEIGHT)

    def _draw_layer_vlines(
        self,
        ax: plt.Axes,
        n_checkpoints: int,
        stride: int = 3,
        color: str = "white",
        lw: float = 1.0,
        alpha: float = 0.6,
        ls: str = "-",
    ) -> None:
        """Draw vertical separator lines at every full-layer boundary."""
        for xi in range(stride, n_checkpoints, stride):
            ax.axvline(xi - 0.5, color=color, lw=lw, ls=ls, alpha=alpha)


    # ── Per-instance plots ─────────────────────────────────────────────────────────

    def plot_logit_lens(
        self,
        sample:       SampleAnly,
        tokenizer,
        mode:        str = "probs",
        # top_down:    bool = False,
        title_prefix: str = "",
        save_path:    Optional[str | Path] = None,
    ) -> plt.Figure:
        """
        Nostalgebraist-style logit-lens heatmap for a single sample.
        
        Each cell shows the **top-1 predicted token** at a given layer, coloured
        by one of four scalar quantities computed from the full-vocabulary
        distribution at that layer.  This is the direct analogue of
        nostalgebraist/transformer-utils ``_plot_logit_lens``.

        Layout
        ------
        - Rows    : checkpoints (emb, L0-attn, …), displayed top-down (final on top). (``top_down=False``).
                    Pass ``top_down=True`` to reverse (embedding at top).
        - Columns : token positions.  Because hidden states here are already
                    last-token-pooled (shape ``(L+1, H)``), there is exactly
                    **one column** per sample.  The bottom x-axis label is the
                    (masked) input representation; the top x-axis shows the
                    final-layer top-1 prediction with a ``*`` prefix.
        - Colour  : controlled by ``mode`` (see below).
        - Text    : top-1 token string at each layer  (``mode="kl"`` shows the
                    KL value instead, matching the reference).
        """

        if mode not in {"probs", "logits", "ranks", "kl"}:
            raise ValueError(f"mode must be one of 'probs','logits','ranks','kl'; got '{mode}'")
        
        # ── 1. Get full-vocab logits & probs ────────────────────────────
        # 1. hs, logits
        hs = sample.all_states.to(torch.float32).cpu() # (C=1+3*L, H) # C = num of checkpoints
        logits = sample.logits.to(torch.float32).cpu() # (C, V)        
        
        # 2. prob dist
        probs_full = np.exp(logits - torch.max(logits, dim=-1, keepdim=True).values ) # np.exp(logits - logits.max(axis=-1, keepdims=True))        
        probs_full /= probs_full.sum(axis=-1, keepdims=True)   # (L+1, vocab)  stable softmax

        # ── 2. Derive display quantities  ─────────        
        # layer_preds[l] = argmax token id at layer l          shape (L+1,)
        layer_preds = logits.argmax(axis=-1)                   # (L+1,)
        # final layer's top-1 token id (scalar)
        final_pred  = layer_preds[-1]

        # Scalar value shown as colour — computed for each layer at the single
        # token position we have (mirrors get_value_at_preds for T=1).
        if mode == "kl":
            # KL( final_layer || layer ) over full vocab
            eps        = 1 / (10 * probs_full.shape[-1])
            final_prob = np.clip(probs_full[[-1]], a_min=eps, a_max=None)  # (1, vocab)
            layer_prob = np.clip(probs_full,        a_min=eps, a_max=None) # (L+1, vocab)
            # sum over vocab → (L+1,), then add a token axis → (L+1, 1)
            to_show = (final_prob * np.log(final_prob / layer_prob)).sum(axis=-1, keepdims=True)
        elif mode == "probs":
            # prob assigned by each layer to the final-layer top-1 token
            to_show = probs_full[:, [final_pred]]              # (L+1, 1)
        elif mode == "logits":
            to_show = logits[:, [final_pred]]                  # (L+1, 1)
        else:  # ranks
            # rank of final_pred in each layer (1 = highest prob)
            to_show = (probs_full >= probs_full[:, [final_pred]]).sum(axis=-1, keepdims=True).astype(float)


        # ── 3. Cell text ──────────────────────────────────────────────────────
        def _tok(idx):
            return repr(tokenizer.decode([int(idx)]))[1:-1]   # strip outer quotes

        if mode == "kl":
            # nostalgebraist uses numeric text for KL mode
            cell_texts = np.array([[f"{to_show[l, 0]:.2f}"] for l in range(len(layer_preds))])
        else:
            cell_texts = np.array([[_tok(layer_preds[l])] for l in range(len(layer_preds))])

        # ── 4. Flip rows (display order) ──────────────────────────────────────
        # Default (top_down=False): final layer at top  ←→  nostalgebraist default
        # to_show = np.array(to_show.tolist()[::-1])
        # cell_texts = np.array(cell_texts.tolist()[::-1])
        n_layers, n_cols = to_show.shape     # n_cols == 1

        # ── 5. Figure & seaborn heatmap ───────────────────────────────────────────
        # Generous height so every layer label is readable
        fig = plt.figure(figsize=(max(4.0, n_cols * 2.5), max(8, n_layers * 0.46)))
        annot_kw: dict = {
            "annot":    cell_texts,
            "fmt":      "",
            "annot_kws": {"size": PLOT_STYLE["annot"], "weight": "bold"},
        }
    
        if mode == "kl":
            annot_kw.update({"cmap": "gray_r", "annot": True, "fmt": ".2f"})
        elif mode == "ranks":
            annot_kw.update({
                "cmap":  "Blues",
                "norm":  mpl.colors.LogNorm(vmin=1, vmax=2000),
                "annot": True,
                "fmt":   ".0f",
            })
        elif mode == "probs":
            annot_kw.update({"cmap": "Blues_r", "vmin": 0, "vmax": 1})
        else:  # logits
            vmin_l = float(np.percentile(to_show, 5))
            vmax_l = float(np.percentile(to_show, 95))
            annot_kw.update({"cmap": "Greys", "vmin": vmin_l, "vmax": vmax_l})
    

        # add font rendering
        heatmap_kw = {k: v for k, v in annot_kw.items()
                      if k not in ("annot", "fmt", "annot_kws")}
        heatmap_kw["annot"] = False
        ax = sns.heatmap(to_show, **heatmap_kw)
        annotate_heatmap_fb(ax, cell_texts[::-1], fontsize=PLOT_STYLE["annot"], fontweight="bold")

        ax.invert_yaxis()

        # Get the colormap and normalizer
        coll   = ax.collections[0]
        cmap   = coll.cmap
        norm   = coll.norm
        
        for text in ax.texts:
            # text position in data coords → which cell row/col
            col_idx = int(text.get_position()[0])   # x → column
            row_idx = int(text.get_position()[1])   # y → row
        
            # Clamp in case of floating-point boundary edge
            col_idx = min(col_idx, to_show.shape[1] - 1)
            row_idx = min(row_idx, to_show.shape[0] - 1)

            # ---------------
            cell_val  = to_show[row_idx, col_idx]
            # data_row_idx = to_show.shape[0] - 1 - row_idx  # ← ADD THIS
            # cell_val = to_show[data_row_idx, col_idx]       # ← use data_row_idx
            # ---------------
            rgba      = cmap(norm(cell_val))         # (R, G, B, A) in [0, 1]
        
            # Perceived luminance (ITU-R BT.709)
            r, g, b = rgba[0], rgba[1], rgba[2]
            luminance = 0.2126 * r + 0.7152 * g + 0.0722 * b
        
            text.set_color("white" if luminance < 0.45 else "black")

            # Resize annotation text after seaborn renders it
            text.set_fontsize(PLOT_STYLE["annot"])
            text.set_fontweight("bold")

        # Colorbar label
        cbar = ax.collections[0].colorbar
        if cbar is not None:
            cbar.ax.tick_params(labelsize=PLOT_STYLE["colorbar"])
            cbar.ax.yaxis.label.set_size(PLOT_STYLE["colorbar"])
            
        # ── 6. Y-axis: layer labels ───────────────────────────────────────────
        # Labels always in natural order (index 0 = embedding, last = final layer).
        # The axis inversion below handles display direction.
        ylabels = SampleAnly.checkpoint_labels(sample.n_layers)   # always natural order
        # ylabels    = ylabels[::-1]
        
        ax.set_yticklabels(ylabels, rotation=0, fontsize=PLOT_STYLE["tick"],
                           fontweight=TICK_WEIGHT)
        ax.set_ylabel("Layer", fontsize=PLOT_STYLE["axis"], fontweight=LABEL_WEIGHT)

        # ── 7. Dual x-axes (mirrors nostalgebraist ax_inputs / ax_targets) ───
        # Bottom axis (ax): input-side label — "last token position"
        ax.set_xticks([0.5])
        ax.set_xticklabels(
            ["[last tok]"], rotation=0,
            fontsize=PLOT_STYLE["tick"], fontweight=TICK_WEIGHT,
        )

        # Top axis: final-layer top-1 prediction, starred
        ax_top = ax.twiny()
        ax_top.set_xlim(ax.get_xlim())
        ax_top.set_xticks([0.5])
        ax_top.set_xticklabels(
            [" "], rotation=0,
            fontsize=PLOT_STYLE["tick"], fontweight=TICK_WEIGHT,
        )
        ax_top.tick_params(axis="x", labelsize=PLOT_STYLE["tick"])
        
        # ── Display order ─────────────────────────────────────────────────────
        # top_down=False (default): final layer on top  → invert so row 0 (embedding) is at bottom
        # top_down=True            : embedding on top   → seaborn's natural order, no invert needed
       

        # ── 8. Title & save ───────────────────────────────────────────────────
        mode_label = {
            "probs": "Probability", "logits": "Logit",
            "ranks": "Rank",        "kl":     "KL divergence",
        }[mode]
        ax.set_title(
            f"{title_prefix}[{mode_label}]  Sample {sample.sample_id}",
            fontsize=PLOT_STYLE["title"],
            fontweight=TITLE_WEIGHT,
            pad=22,
        )
        self._apply_axis_style(ax)
        fig.tight_layout()
    
        if save_path:
            fig.savefig(save_path, dpi=150, bbox_inches="tight")
        return fig

    
    def plot_sublayer_trajectory(
        self,
        sample:    SampleAnly,
        tokenizer: PreTrainedTokenizerBase,
        save_path: Optional[str | Path] = None,
    ) -> plt.Figure:
        """
        Per-instance trajectory with two panels:
          Top strip : language colour bar at every checkpoint.
          Bottom    : top-1 token text table.
        """
        C        = sample.n_checkpoints
        labels   = sample.cp_labels
        cp_range = np.arange(C)
    
        token_ids = sample.max_prob_token_ids.tolist()

        # ── Build sanitized token strings ────────────────────────────────────────
        tok_strs = [
        sanitize_tok_str(
            repr(tokenizer.decode([int(tid)]))[1:-1]   # strip outer quotes
        )
        for tid in token_ids
        ]

        lang_ids  = sample.top1_lang_ids.tolist()
        lang_lbls = [ALL_LANG_LABELS[li] for li in lang_ids]
        lang_cols = [LANG_COLORS[l] for l in lang_lbls]
    
        fig, axes = plt.subplots(
            2, 1,                                          # <-- 2 panels, not 3
            figsize=(max(16, C * 0.44), 6),                # <-- slightly shorter
            gridspec_kw={"height_ratios": [0.6, 1.0]},    # <-- drop middle ratio
        )
        fig.suptitle(
            f"Sub-layer trajectory  |  Sample {sample.sample_id}",
            fontsize=PLOT_STYLE["title"],
            fontweight=TITLE_WEIGHT,
        )
    
        # ── Top strip: language colour bar ────────────────────────────────────────
        ax_strip = axes[0]
        for xi, (col, lbl) in enumerate(zip(lang_cols, lang_lbls)):
            ax_strip.bar(xi, 1, color=col, width=0.95)
    
        ax_strip.set_xlim(-0.5, C - 0.5)
        ax_strip.set_ylim(0, 1)
        ax_strip.set_yticks([])
        ax_strip.set_xticks([])
        ax_strip.set_title(
            "Top-1 token language",
            fontsize=PLOT_STYLE["axis"],
            fontweight=TITLE_WEIGHT,
            pad=3,
        )
    
        seen_langs = list(dict.fromkeys(lang_lbls))
        patches    = [mpl.patches.Patch(color=LANG_COLORS[l], label=l) for l in seen_langs]
        ax_strip.legend(
            handles=patches,
            loc="upper right",
            fontsize=PLOT_STYLE["legend"],
            ncol=min(4, len(seen_langs)),
            framealpha=0.90,
            edgecolor="0.7",
        )
        self._draw_layer_vlines(ax_strip, C, color="white", lw=0.9, alpha=0.65)
    
        # ── Bottom: top-1 token text table ────────────────────────────────────────
        ax_tok = axes[1]                                   # <-- was axes[2]
        ax_tok.set_xlim(-0.5, C - 0.5)
        ax_tok.set_ylim(0, 1)
        ax_tok.set_yticks([])
    
        for xi, (ts, col) in enumerate(zip(tok_strs, lang_cols)):
            draw_rotated_token(
                ax_tok,
                x  = xi,
                y  = 0.85, # 0.02,                  # just above axis line
                text = ts,
                fontsize = PLOT_STYLE["strip"] + 1,
                rotation = 70, # 45
                bbox_kw  = dict(boxstyle="round,pad=0.15", fc=col, alpha=0.28, lw=0),
                fontweight = "bold",
            )

        self._draw_layer_vlines(ax_tok, C, color="black", lw=0.6, ls="--", alpha=0.4)
        ax_tok.set_xticks(cp_range)
        ax_tok.set_xticklabels(
            labels, rotation=70, # 45
            fontsize=PLOT_STYLE["strip"],
            fontweight=TICK_WEIGHT,
            ha="right",
        )
        
        ax_tok.set_title(
            "Top-1 token per checkpoint",
            fontsize=PLOT_STYLE["axis"],
            fontweight=TITLE_WEIGHT,
            pad = 4,        # extra gap between spine and tick label baseline
        )
    
        fig.tight_layout()
        if save_path:
            fig.savefig(save_path, dpi=150, bbox_inches="tight")
        return fig

        
    

    # ── Population-level plots ─────────────────────────────────────────────────────
    def plot_lang_proportion(
        self,
        samples:   List,                     # List[SampleAnly]
        show_grid: bool = True,
        title:     str  = "",
        save_path: Optional[str | Path] = None,
    ) -> plt.Figure:
        """
        Stacked-bar chart of language proportions across all checkpoints.
    
        Vertical dashed lines mark full transformer-layer boundaries.
        """
        proportions = self.layerwise_lang_proportions(samples)   # (C, n_lang)
        C           = proportions.shape[0]
        cp_labels   = SampleAnly.checkpoint_labels(samples[0].n_layers)
        x           = np.arange(C)
    
        fig, ax = plt.subplots(figsize=(max(16, C * 0.44), 5.5))
    
        bottom = np.zeros(C)
        for li, lbl in enumerate(ALL_LANG_LABELS):
            heights = proportions[:, li] # ---------> 레이어별 bar chart 그리는 거 맞는지?
            ax.bar(x, heights, bottom=bottom, color=LANG_COLORS[lbl],
                   label=lbl, width=0.92, edgecolor="none")
            bottom += heights
    
        self._draw_layer_vlines(ax, C)
    
        ax.set_xlim(-0.5, C - 0.5)
        ax.set_ylim(0, 1)
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
        ax.set_ylabel("Proportion of samples",
                      fontsize=PLOT_STYLE["axis"], fontweight=LABEL_WEIGHT)
        ax.set_xlabel("Checkpoint",
                      fontsize=PLOT_STYLE["axis"], fontweight=LABEL_WEIGHT)
        ax.set_xticks(x)
        ax.set_xticklabels(
            cp_labels, rotation=70,
            fontsize=PLOT_STYLE["tick"],
            fontweight=TICK_WEIGHT,
            ha="right",
        )
    
        ax.legend(
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
            borderaxespad=0,
            fontsize=PLOT_STYLE["legend"],
            frameon=True,
            framealpha=0.92,
            edgecolor="0.7",
            ncol=min(4, len(ALL_LANG_LABELS)),
        )
        fig.subplots_adjust(right=0.82)
    
        if show_grid:
            ax.grid(axis="y", ls=":", alpha=0.35, zorder=0)
    
        ax.set_title(
            title or f"Latent-language proportions  (N={len(samples)})",
            fontsize=PLOT_STYLE["title"],
            fontweight=TITLE_WEIGHT,
        )
        self._apply_axis_style(ax)
        fig.tight_layout()
    
        if save_path:
            fig.savefig(save_path, dpi=150, bbox_inches="tight")
        return fig




    def plot_lang_proportion_per_slot(
            self,
            samples:   List,
            title:     str  = "",
            save_path: Optional[str | Path] = None,
        ) -> plt.Figure:
            proportions = self.layerwise_lang_proportions(samples)
            cp_labels   = SampleAnly.checkpoint_labels(samples[0].n_layers)
            L           = samples[0].n_layers
    
            emb_idx  = [0]
            attn_idx = [1 + li * 3 + SLOT_POST_ATTN  for li in range(L)]
            mlp_idx  = [1 + li * 3 + SLOT_POST_MLP   for li in range(L)]
            blk_idx  = [1 + li * 3 + SLOT_FULL_BLOCK for li in range(L)]
    
            groups = [
                ("Embedding",      emb_idx),
                ("Post-attention", attn_idx),
                ("Post-MLP",       mlp_idx),
                ("Full-block",     blk_idx),
            ]
    
            # ── Sizing ────────────────────────────────────────────────────────────────
            BAR_W       = 0.70
            COL_W_PER_L = 0.32          # inches per layer for the 3 large panels
            EMB_W       = 0.55          # fixed width for the single embedding panel
            PANEL_W     = max(3.0, L * COL_W_PER_L)
            FIG_W       = EMB_W + 3 * PANEL_W + 0.4   # no side legend needed
            FIG_H       = 5.0           # taller so large fonts don't crowd panel titles
    
            fig, axes = plt.subplots(
                1, 4,
                figsize=(FIG_W, FIG_H),
                gridspec_kw={"width_ratios": [EMB_W, PANEL_W, PANEL_W, PANEL_W]},
            )
    
            # Reserve vertical room for the suptitle so it never overlaps panel titles.
            # rect=[left, bottom, right, top] — shrink the top of the layout area.
            SUPTITLE_Y   = 0.93 # 0.97          # normalized figure coords
            LAYOUT_TOP   = 0.91 # 0.88          # tight_layout stops here; gap above = suptitle space
            LAYOUT_BOT   = 0.14          # bottom gap reserved for horizontal legend
    
            fig.suptitle(
                f"[{title}] Latent-language by sub-layer slot  (N={len(samples)})",
                fontsize=PLOT_STYLE["title"],
                fontweight=TITLE_WEIGHT,
                y=SUPTITLE_Y,
            )

            # Strip the redundant sublayer suffix from x-tick labels (e.g. "L3-attn" → "L3")
            def _strip_sublayer(label: str) -> str:
                return label.split("-")[0] or "Emb"
        
            for ax, (panel_title, idx_list) in zip(axes, groups):
                sub_prop   = proportions[idx_list]
                sub_labels = [_strip_sublayer(cp_labels[i]) for i in idx_list]
                x          = np.arange(len(idx_list))
                bottom     = np.zeros(len(idx_list))
    
                for li, lbl in enumerate(ALL_LANG_LABELS):
                    ax.bar(x, sub_prop[:, li], bottom=bottom,
                           color=LANG_COLORS[lbl], label=lbl,
                           width=BAR_W, edgecolor="none")
                    bottom += sub_prop[:, li]
    
                ax.set_ylim(0, 1)
                ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
                ax.set_xticks(x)
                ax.set_xticklabels(
                    sub_labels, rotation=70,
                    fontsize=PLOT_STYLE["tick"],
                    fontweight=TICK_WEIGHT,
                    ha="right",
                    rotation_mode="anchor",   # ← anchors rotation pivot to the tick point
                )
                ax.set_title(panel_title,
                             fontsize=PLOT_STYLE["axis"], fontweight=TITLE_WEIGHT, pad=4)
                ax.grid(axis="y", ls=":", alpha=0.35, zorder=0)
    
                # Y-label + ticks only on leftmost panel
                if ax is axes[0]:
                    ax.set_ylabel("Proportion",
                                  fontsize=PLOT_STYLE["axis"], fontweight=LABEL_WEIGHT)
                else:
                    ax.set_yticks([])    # remove ticks AND labels (saves more space than yticklabels=[])
    
                self._apply_axis_style(ax)
    
            # ── Horizontal figure-level legend (below all panels) ─────────────────
            handles, labels = axes[0].get_legend_handles_labels()
            fig.legend(
                handles, labels,
                loc="lower center",
                bbox_to_anchor=(0.5, 0),     # centred at the very bottom of the figure
                ncol=len(ALL_LANG_LABELS),   # all entries in one row
                fontsize=PLOT_STYLE["legend"],
                frameon=True,
                framealpha=0.92,
                edgecolor="0.7",
                handlelength=0.8,
                handletextpad=0.3,
                columnspacing=0.8,
            )
    
            fig.tight_layout(
                rect=[0, LAYOUT_BOT, 1, LAYOUT_TOP],  # gaps for legend (bottom) and suptitle (top)
                w_pad=0.4,
                h_pad=0.0,
            )
    
            if save_path:
                fig.savefig(save_path, dpi=300, bbox_inches="tight")
            return fig
        

    def plot_lang_switching(
        self,
        samples:   List,
        title:     str  = "",
        save_path: Optional[str | Path] = None,
    ) -> plt.Figure:
        """
        Line plot of language-switching rate between consecutive checkpoints.
    
        High values indicate abrupt representational change (many samples flip
        their top-1 token language at that transition).
        """
        switch_rate  = self.lang_switching_rate(samples)    # (C-1,)
        C            = samples[0].n_checkpoints
        cp_labels    = SampleAnly.checkpoint_labels(samples[0].n_layers)
        trans_labels = [f"{cp_labels[i]}→{cp_labels[i+1]}" for i in range(C - 1)]
        x            = np.arange(C - 1)
    
        fig, ax = plt.subplots(figsize=(max(16, (C - 1) * 0.44), 5.0))
        ax.plot(x, switch_rate, color="darkorange", lw=2.5, marker="o", ms=4,
                markerfacecolor="darkorange", markeredgewidth=0)
        ax.fill_between(x, 0, switch_rate, color="darkorange", alpha=0.15)
    
        for xi in range(2, C - 1, 3):
            ax.axvline(xi - 0.5, color="grey", lw=0.6, ls="--", alpha=0.4)
    
        ax.set_xlim(-0.5, C - 1.5)
        ax.set_ylim(0, 1)
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
        ax.set_ylabel(
            "Fraction of samples switching language",
            fontsize=PLOT_STYLE["axis"], fontweight=LABEL_WEIGHT,
        )
        ax.set_xlabel(
            "Checkpoint transition",
            fontsize=PLOT_STYLE["axis"], fontweight=LABEL_WEIGHT,
        )
        ax.set_xticks(x)
        ax.set_xticklabels(
            trans_labels, rotation=70,
            fontsize=PLOT_STYLE["tick"],
            fontweight=TICK_WEIGHT,
            ha="right",
        )
        ax.set_title(
            title or f"Language switching rate  (N={len(samples)})",
            fontsize=PLOT_STYLE["title"], fontweight=TITLE_WEIGHT,
        )
        ax.grid(axis="y", ls=":", alpha=0.4)
        self._apply_axis_style(ax)
        fig.tight_layout()
    
        if save_path:
            fig.savefig(save_path, dpi=150, bbox_inches="tight")
        return fig



    def plot_dominant_language_heatmap(
        self,
        samples:   List,
        title:     str  = "",
        save_path: Optional[str | Path] = None,
    ) -> plt.Figure:
        """
        Heatmap of language proportions: rows = languages, columns = checkpoints.
    
        Rows are sorted by their peak proportion; languages with zero presence
        are omitted.  Annotated cells show the proportion value where ≥ 0.01.
        """
        proportions = self.layerwise_lang_proportions(samples)   # (C, n_lang)
        C           = proportions.shape[0]
        cp_labels   = SampleAnly.checkpoint_labels(samples[0].n_layers)
    
        # Filter to languages that appear at all
        present = [
            (li, lbl) for li, lbl in enumerate(ALL_LANG_LABELS)
            if proportions[:, li].max() > 0.0
        ]
        if not present:
            fig, ax = plt.subplots()
            ax.text(0.5, 0.5, "No language data.",
                    ha="center", va="center", fontsize=PLOT_STYLE["axis"])
            return fig
    
        lang_indices = [li for li, _ in present]
        lang_labels  = [lbl for _, lbl in present]
        data         = proportions[:, lang_indices].T          # (n_lang, C)
    
        # Sort by peak proportion (most dominant first)
        order       = np.argsort(data.max(axis=1))[::-1]
        data        = data[order]
        lang_labels = [lang_labels[i] for i in order]
    
        fig, ax = plt.subplots(
            figsize=(max(16, C * 0.44), max(4, len(lang_labels) * 0.70))
        )
        im = ax.imshow(data, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)
    
        cbar = plt.colorbar(im, ax=ax, label="Proportion")
        cbar.ax.tick_params(labelsize=PLOT_STYLE["colorbar"])
        cbar.ax.yaxis.label.set_size(PLOT_STYLE["colorbar"])
        cbar.ax.yaxis.label.set_fontweight(LABEL_WEIGHT)
    
        ax.set_xticks(range(C))
        ax.set_xticklabels(
            cp_labels, rotation=70,
            fontsize=PLOT_STYLE["tick"],
            fontweight=TICK_WEIGHT,
            ha="right",
        )
        ax.set_yticks(range(len(lang_labels)))
        ax.set_yticklabels(
            lang_labels,
            fontsize=PLOT_STYLE["axis"],
            fontweight=TICK_WEIGHT,
        )
        ax.set_xlabel("Checkpoint",
                      fontsize=PLOT_STYLE["axis"], fontweight=LABEL_WEIGHT)
        ax.set_title(
            title or f"Language proportion heatmap  (N={len(samples)})",
            fontsize=PLOT_STYLE["title"], fontweight=TITLE_WEIGHT,
        )
    
        for row in range(len(lang_labels)):
            for col in range(C):
                val = data[row, col]
                if val >= 0.01:
                    ax.text(
                        col, row, f"{val:.2f}",
                        ha="center", va="center",
                        fontsize=PLOT_STYLE["annot"],
                        fontweight="bold",
                        color="white" if val > 0.60 else "black",
                    )
    
        # Layer boundary separators
        self._draw_layer_vlines(ax, C, color="white", lw=0.9, alpha=0.75)
    
        self._apply_axis_style(ax)
        fig.tight_layout()
    
        if save_path:
            fig.savefig(save_path, dpi=150, bbox_inches="tight")
        return fig


    # ── In LogitLensAnalyzer ──────────────────────────────────────────────────────
    def release(self) -> None:
        """Explicitly release GPU references held by the analyzer."""
        # norm_layer and lm_head hold model weight references
        self.norm_layer = None
        self.lm_head    = None
        self.lang_id_lookup = self.lang_id_lookup.cpu()

In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# 8.  Model helpers
# ─────────────────────────────────────────────────────────────────────────────

def _get_norm_and_head(model: nn.Module) -> Tuple[nn.Module, nn.Module]:
    multimodal_prefixes = ("gemma3", "gemma4", "qwen3_5")
    if any(model.config.model_type.startswith(p) for p in multimodal_prefixes):
        norm_layer = model.model.language_model.norm
    else:
        norm_layer = model.model.norm
    return norm_layer, model.lm_head


def load_model(model_name: str, device: str) -> nn.Module:
    kwargs: Dict = dict(
        torch_dtype="auto",
        device_map="auto" if device == "cuda" else None,
        local_files_only=True,
    )

    return AutoModelForCausalLM.from_pretrained(model_name, **kwargs)


def build_analyzer(
    model:      nn.Module,
    tokenizer:  PreTrainedTokenizerBase,
    device:     str = "cuda",
    batch_size: int = 32,
) -> LogitLensAnalyzer:
    """
    Construct a LogitLensAnalyzer from an already-loaded model and tokenizer.

    Builds the token→language map once (a few seconds for large vocabularies).
    """
    norm_layer, lm_head  = _get_norm_and_head(model)
    
    print("Building token→language map …")
    token_lang_map = build_token_language_map(tokenizer)
    print(f"  vocab size: {len(token_lang_map)}")

    return LogitLensAnalyzer(
        norm_layer=norm_layer,
        lm_head=lm_head,
        token_lang_map=token_lang_map,
        device=device,
        batch_size=batch_size,
    )


# ─────────────────────────────────────────────────────────────────────────────
# 9.  Dataset loaders
# ─────────────────────────────────────────────────────────────────────────────

def save_bactrian_x_dataset(languages: list[str], max_samples: int | None = None, seed: int = 42):
    """
    Load Bactrian-X subsets for multiple languages, sampling the same IDs across all.

    Args:
        languages:   list of language names, e.g. ["korean", "chinese", "french", "bengali"]
        max_samples: if set, sample this many examples (same IDs for all languages)
        seed:        random seed for reproducibility

    Returns:
        dict[str, Dataset] — one dataset per language, aligned by id
    """

    # ── 1. Load each language by the sampled IDs ─────────────────
    result = {}
    load_bac_lang = lambda lang: load_dataset("json", data_files=f"hf://datasets/MBZUAI/Bactrian-X/data/{lang}.json.gz",
                                              split="train")
    for lang_name in languages:
        path = f"./bactrian_{lang_name}"
        try:
            ds = load_from_disk(path)
            result[lang_name] = ds
        except:
            # 1. load dataset
            ds = load_bac_lang(lang_name)
            ds = ds.sort("id") # restore deterministic order

            # 2. train/valid/test split
            ds_train = ds.train_test_split(test_size=0.2, seed=seed)
            ds_valid = ds_train['test'].train_test_split(test_size=0.5, seed=seed)
            ds = DatasetDict({
                'train': ds_train['train'],
                'test': ds_valid['test'],
                'validation': ds_valid['train']
                })
            ds.save_to_disk(path) # split 상태 저장
            result[lang_name] = ds
            
        print("-"*30 + lang_name)
        print(ds)
        
    return result  # {"korean": Dataset, "chinese": Dataset, ...}


def format_prompt(row: dict, lang: str = "en") -> str:
    INSTRUCTION_STRS = {
        "en": "Instruction",
        "zh": "指令",
        "ko": "지시",
        "te": "సూచన",
        "fr": "Instruction",
        "pt": "Instrução",
        "sw": "Maelekezo",
        "ja": "指示",
        "my": "ညွှန်ကြားချက်",
        "ta": "அறிவுறுத்தல்",
    }
    
    INPUT_STRS = {
        "en": "Input",
        "zh": "输入",
        "ko": "입력",
        "te": "ఇన్‌పుట్",
        "fr": "Entrée",
        "pt": "Entrada",
        "sw": "Ingizo",
        "ja": "入力",
        "my": "ထည့်သွင်းချက်",
        "ta": "உள்ளீடு",
    }
    
    OUTPUT_STRS = {
        "en": "Output",
        "zh": "输出",
        "ko": "출력",
        "te": "అవుట్‌పుట్",
        "fr": "Sortie",
        "pt": "Saída",
        "sw": "Tokeo",
        "ja": "出力",
        "my": "ထွက်ချက်",
        "ta": "வெளியீடு",
    }

    instruction = row.get("instruction", "")
    input_text  = row.get("input", "")
    output_text = row.get("output", "")

    instr_key  = INSTRUCTION_STRS[lang]
    input_key  = INPUT_STRS[lang]
    output_key = OUTPUT_STRS[lang]

    if input_text:
        # Three-column case: instruction + input + output
        question_string =         (
            f"{instr_key}: {instruction}\n"
            f"{input_key}: {input_text}"
        )
        prompt = (
            f"{instr_key}: {instruction}\n"
            f"{input_key}: {input_text}\n"
            f"{output_key}: "
        )
    else:
        # Two-column case: instruction + output only
        question_string =         (
            f"{instr_key}: {instruction}"
        )
        prompt = (
            f"{instr_key}: {instruction}\n"
            f"{output_key}: "
        )

    msg = [ { "role" : "user", "content" : prompt } ]
    return msg, question_string



def load_bactrian_x_dataset(
    language:    str,
    tokenizer:  PreTrainedTokenizerBase,
    split:       str = "train",
    max_samples: Optional[int] = None,
    seed: int = 42,
    prompt_fn:   Optional[Callable[[Dict], str]] = None,
) -> List[Dict]:
    """
    Load CohereLabs/aya_collection_language_split and convert to the dict
    schema expected by capture_hidden_states_batched().

    Parameters
    ----------
    language    : language config name, e.g. "korean", "arabic", "english"
    split       : "train", "test", or "validation"
    max_samples : cap (useful for debugging)
    prompt_fn   : (row: dict) → str; defaults to ``row["inputs"]``

    Returns
    -------
    list[dict] with keys:
        "sample_id", "question", "answer", "language",
    """

    # load train split of target language subset from bactrian-x
    try:
        from datasets import load_from_disk
        path = f"./bactrian_{language}/train"
        ds = load_from_disk(path)
    except:
        ds = save_bactrian_x_dataset(languages=[language], max_samples=max_samples, seed=42)
        ds = ds["train"]

    # filter train set by sampled ids
    all_ids = ds["id"]  # e.g. ["dolly-10031", "alpaca-19320", ...]
    if max_samples is not None:
        rng = random.Random(seed)
        sampled_ids = set(rng.sample(all_ids, min(max_samples, len(all_ids))))
    else:
        sampled_ids = set(all_ids)
    
    ds = ds.filter(lambda ex: ex["id"] in sampled_ids) # filter the dataset by ids

    def tok_msg(row):
        msg, question_string = format_prompt(row, language)
        result = tokenizer.apply_chat_template( 
                 msg, tokenize=False, add_generation_prompt=True, enable_thinking=False,
            )
        return result, question_string

    print(ds)
    print("use train split only:", "len(ds)", len(ds))
    print("ds[0]", ds[0])
    print("prompt", tok_msg(ds[0])[0])
    print("question", tok_msg(ds[0])[1])

    return [
        {
            "sample_id":    str(row.get("id", i)), # sample_source, script, dataset_name, task_type 제거
            "question":     tok_msg(row)[1], # instruction + input (w/o chat template)
            "answer":      row.get("output", ""), # answer
            "prompt" :     tok_msg(row)[0], # instruction + input (w/ chat template)
            "language":     row.get("language", language),
        }
        for i, row in enumerate(ds)
    ]


# ─────────────────────────────────────────────────────────────────────────────
# 10.  End-to-end pipeline
# ─────────────────────────────────────────────────────────────────────────────

def analyze_bactrian_x(
    path_model:  str,
    language:    str,
    padding_type: str,
    split:       str = "train",
    max_samples: Optional[int] = None,
    batch_size:  int = 32,
    seed:  int = 42,
    out_pkl:     Optional[str | Path] = None,
    device:      str = "cuda",
) -> Tuple["LogitLensAnalyzer", List[SampleAnly], PreTrainedTokenizerBase]:
    """
    Full pipeline:
      load model → load aya dataset → capture hidden states (batched) →
      run logit lens → return (analyzer, samples, tokenizer).

    Parameters
    ----------
    path_model  : HuggingFace model name or local path
    language    : aya language config, e.g. "korean"
    split       : "train" | "test" | "validation"
    max_samples : cap for quick experiments
    batch_size  : used for both capture and unembedding
    out_pkl     : if given, captured hidden states are cached here
    device      : "cuda" or "cpu"
    """
    model     = load_model(path_model, device)
    tokenizer = AutoTokenizer.from_pretrained(path_model)
    if tokenizer.pad_token is None:
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "left"

    analyzer = build_analyzer(model, tokenizer, device=device, batch_size=batch_size)
    dataset  = load_bactrian_x_dataset(language, tokenizer, split=split, max_samples=max_samples, seed=seed)
    

    results = capture_hidden_states_batched(
        model,
        tokenizer,
        dataset,
        padding_type=padding_type,
        device=device,
        capture_batch_size=batch_size,
        out_path=out_pkl,
    )

    
    # Build SampleAnly objects from in-memory results (or reload from pkl)
    if out_pkl:
        samples = analyzer.load_samples(out_pkl, dtype=model.dtype)
    
    else:
        samples = [
            SampleAnly(
                sample_id=d["sample_id"],
                question=d["question"],
                answer=d.get("answer", ""),
                prompt=d["prompt"],
                logits=d.get("logits", torch.tensor([])),
                language=d.get("language"),
                embedding_state=torch.as_tensor(d["embedding_state"]),
                layer_states=torch.as_tensor(d["layer_states"]),
            )
            for d in results
        ]
    
    analyzer.process_hidden_states(samples)
    return analyzer, samples, tokenizer

In [13]:

# ─────────────────────────────────────────────────────────────────────────────
# 12.  Convenience figure export
# ─────────────────────────────────────────────────────────────────────────────

def save_figs(
    analyzer:  LogitLensAnalyzer,
    samples:   List[SampleAnly],
    tokenizer: PreTrainedTokenizerBase,
    prefix:    str,
    ids: List[int],
    title,
) -> List[plt.Figure]:
    """
    Generate and save the standard suite of analysis figures.

    Saved files
    -----------
    {prefix}_lang_proportion.png
    {prefix}_lang_per_slot.png
    {prefix}_lang_heatmap.png
    {prefix}_lang_switching.png
    {prefix}_traj_first.png
    {prefix}_traj_last.png

    Returns
    -------
    List of Figure objects in the same order.
    """
    figs = []

    def _save(fig: plt.Figure, tag: str) -> plt.Figure:
        path = f"{prefix}_{tag}.png"
        fig.savefig(path, dpi=150, bbox_inches="tight")
        print(f"Saved: {path}")
        figs.append(fig)
        return fig

    # ---------- logit lens anly per sample
    # for i in ids:
    #     _save(analyzer.plot_sublayer_trajectory(samples[i], tokenizer), f"traj_{i}")
    #     _save(analyzer.plot_logit_lens(samples[i], tokenizer), f"logit_lens_{i}")
        

    # ---------- anly over all samples
    # _save(analyzer.plot_lang_proportion(samples), "lang_proportion")
    _save(analyzer.plot_lang_proportion_per_slot(samples, title=title), "lang_per_slot")
    # _save(analyzer.plot_lang_switching(samples), "lang_switching")
    # _save(analyzer.plot_dominant_language_heatmap(samples), "lang_heatmap")

    return figs

In [14]:
def run_exp(n, bsz, device, ids, padding_type, path_model, model_alias, run_mode, language):
    prefix = f"bac-x_{language}/bac-x_{language}_{padding_type}_{model_alias}_{n}_{bsz}" # llama3.1-8b # path_model="meta-llama/Llama-3.1-8B-Instruct"
    out_pkl=f"{prefix}.pkl"

    print("prefix", prefix)
    
    model     = load_model(path_model, device)
    print(model)
    
    tokenizer = AutoTokenizer.from_pretrained(path_model)
    if tokenizer.pad_token is None:
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "left"

    if run_mode == "scratch":
        analyzer, samples, tokenizer = analyze_bactrian_x(
            path_model=path_model,
            language=language, #"korean",
            padding_type=padding_type,
            split="train",
            max_samples=n,
            batch_size=bsz,
            seed=42,
            out_pkl=f"{prefix}.pkl",
            device="cuda",
        )
    elif run_mode == "pkl":
        analyzer = build_analyzer(model, tokenizer, device=device, batch_size=bsz)
        samples = analyzer.load_samples(out_pkl, dtype=model.dtype)
        analyzer.process_hidden_states(samples)
        
    figs = save_figs(analyzer, samples, tokenizer, prefix=prefix, ids=ids, title=model_alias)

    # ---------------------
    # 1. Release analyzer's refs to model submodules first
    analyzer.release()
    
    # 2. Clear sample tensors (logits are large)
    for s in samples:
        s.logits              = torch.tensor([])
        s.embedding_state     = torch.tensor([])
        s.layer_states        = torch.tensor([])
        s.max_prob_token_ids  = torch.tensor([])
        s.top1_lang_ids       = torch.tensor([])
    
    # 3. Now delete everything
    del analyzer, samples, tokenizer, model
    
    # 4. GC + cache clear
    gc.collect()
    torch.cuda.empty_cache()
    
    # 5. Verify
    print(torch.cuda.memory_allocated() / 1e9, "GB still allocated")
    print(torch.cuda.memory_reserved()  / 1e9, "GB still reserved")

# Experiments

In [15]:
# languages = ["zh", "ko", "te" ]
# datasets = save_bactrian_x_dataset(languages, max_samples=5000, seed=42)

In [16]:
n = 5000
bsz = 32
device="cuda"
ids = [204, 912, 1828, 2006, 2253] # sorted(random.sample(range(n), 5))
padding_type="last-literal" # "last-non-pad"
print("sample ids for logit lens", ids)

from functools import partial

run_exp_last_literal = partial(
    run_exp,
    n=n, bsz=bsz, device=device, ids=ids, padding_type="last-literal"
)

sample ids for logit lens [204, 912, 1828, 2006, 2253]


In [19]:
def run_all_exp_last_literal(language, run_mode="scratch"):
    # baseline
    run_exp_last_literal(path_model="meta-llama/Llama-3.2-3B-Instruct", model_alias="llama3.2-3b-it", run_mode=run_mode, language=language)
    run_exp_last_literal(path_model="google/gemma-2-2b-it", model_alias="gemma-2-2b-it", run_mode=run_mode, language=language)
    
    # finetuned
    run_exp_last_literal(path_model=f"../may_merged/llama3.2-3b-it__{language}__all_rank8", model_alias=f"peft_llama32_{language}", run_mode=run_mode, language=language)
    run_exp_last_literal(path_model=f"../may_merged/gemma-2-2b-it__{language}", model_alias=f"peft_gemma2_{language}", run_mode=run_mode, language=language)

In [1]:
run_all_exp_last_literal("zh")

In [2]:
run_all_exp_last_literal("ko")

In [3]:
run_all_exp_last_literal("te")